In [1]:
import sys
print(sys.version)




3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]


In [2]:
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
CUDA available: True


In [3]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device} | CUDA available: {torch.cuda.is_available()}")

# Clear GPU memory if you re-run cells often
torch.cuda.empty_cache()
gc.collect()


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Using device: cuda | CUDA available: True


20

In [4]:
from transformers import AutoTokenizer, AutoModel

bert_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_encoder = AutoModel.from_pretrained(bert_name).to(device)

# Small test
text = "VedhLLM combines symbolic and semantic reasoning."
inputs = bert_tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = bert_encoder(**inputs, return_dict=True)

print("✅ Encoder hidden state shape:", outputs.last_hidden_state.shape)


c:\Users\vedhr\CODES\LLM\.venv\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

c:\Users\vedhr\CODES\LLM\.venv\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✅ Encoder hidden state shape: torch.Size([1, 12, 768])


In [5]:
import torch.nn as nn

class Bridge(nn.Module):
    def __init__(self, enc_dim=768, dec_dim=4096, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(enc_dim, dec_dim)
        self.norm = nn.LayerNorm(dec_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.proj(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x.to(device)

bridge = Bridge().to(device)
proj = bridge(outputs.last_hidden_state)
print("✅ Projected bridge shape:", proj.shape)


✅ Projected bridge shape: torch.Size([1, 12, 4096])


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

llama_name = "meta-llama/Llama-2-7b-chat-hf"  # or "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

llama_tokenizer = AutoTokenizer.from_pretrained(llama_name, use_fast=True)
llama_model = AutoModelForCausalLM.from_pretrained(
    llama_name,
    quantization_config=bnb_config,
    device_map="auto"
)

llama_tokenizer.pad_token = llama_tokenizer.eos_token

print("✅ LLaMA loaded successfully in 4-bit mode")


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-chat-hf.
401 Client Error. (Request ID: Root=1-69142198-0d989a5616933c853da3a78d;466cb51a-fd07-4e6f-9f29-bed75aff88c3)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-chat-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-chat-hf is restricted. You must have access to it and be authenticated to access it. Please log in.

In [4]:
from transformers import AutoModel, AutoTokenizer
import torch

# Use a strong BERT encoder
encoder_name = "bert-large-uncased"
encoder_tokenizer = AutoTokenizer.from_pretrained(encoder_name)
encoder_model = AutoModel.from_pretrained(encoder_name)

print("✅ BERT encoder loaded")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

c:\Users\vedhr\CODES\LLM\.venv\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

✅ BERT encoder loaded


In [5]:
import torch.nn as nn

class BridgeLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.bridge = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim)
        )
    
    def forward(self, x):
        return self.bridge(x)

# Example bridge from BERT (1024d) → TinyLLaMA (4096d)
bridge = BridgeLayer(1024, 4096)
print("✅ Bridge initialized (BERT → Decoder)")


✅ Bridge initialized (BERT → Decoder)


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 🦙 Use TinyLLaMA (fits comfortably on 8GB VRAM)
llama_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ⚙️ Quantization config — runs in 4-bit on consumer GPUs
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 🔡 Load tokenizer + model
decoder_tokenizer = AutoTokenizer.from_pretrained(llama_name, use_fast=True)
decoder_model = AutoModelForCausalLM.from_pretrained(
    llama_name,
    quantization_config=bnb_config,
    device_map="auto"
)

decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

print("✅ TinyLLaMA decoder loaded in 4-bit mode")


tokenizer_config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


model.safetensors:  53%|#####2    | 1.16G/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [6]:
text = "VedhLLM is a hybrid model combining symbolic and semantic reasoning."
inputs = encoder_tokenizer(text, return_tensors="pt")

# 1️⃣ Encode
with torch.no_grad():
    enc_out = encoder_model(**inputs).last_hidden_state.mean(dim=1)

# 2️⃣ Bridge
bridged = bridge(enc_out)

# 3️⃣ Decode
prompt = "Summarize this concept: "
input_ids = decoder_tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Add bridged embedding to model as context (simulated fusion)
outputs = decoder_model.generate(
    input_ids,
    max_new_tokens=60,
    temperature=0.7,
    do_sample=True
)

print("\n💬 Output:")
print(decoder_tokenizer.decode(outputs[0], skip_special_tokens=True))


NameError: name 'decoder_tokenizer' is not defined

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

llama_name = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

decoder_tokenizer = AutoTokenizer.from_pretrained(llama_name, use_fast=True, token=True)
decoder_model = AutoModelForCausalLM.from_pretrained(
    llama_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=True
)

decoder_tokenizer.pad_token = decoder_tokenizer.eos_token
print("✅ LLaMA 2 7B loaded successfully in 4-bit mode")


LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

llama_name = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

decoder_tokenizer = AutoTokenizer.from_pretrained(llama_name, use_fast=True)
decoder_model = AutoModelForCausalLM.from_pretrained(
    llama_name,
    quantization_config=bnb_config,
    device_map="auto"
)

decoder_tokenizer.pad_token = decoder_tokenizer.eos_token
print("✅ LLaMA 2 7B loaded successfully in 4-bit mode")


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Error while downloading from https://huggingface.co/meta-llama/Llama-2-7b-chat-hf/resolve/main/model-00002-of-00002.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


model-00002-of-00002.safetensors:  64%|######3   | 2.22G/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

llama_name = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

decoder_tokenizer = AutoTokenizer.from_pretrained(llama_name, use_fast=True)
decoder_model = AutoModelForCausalLM.from_pretrained(
    llama_name,
    quantization_config=bnb_config,
    device_map="auto"
)

decoder_tokenizer.pad_token = decoder_tokenizer.eos_token
print("✅ LLaMA 2 7B loaded successfully in 4-bit mode")

# Test prompt (✔ NO .to("cuda")!)
prompt = "Explain the importance of hybrid reasoning in AI."
inputs = decoder_tokenizer(prompt, return_tensors="pt").to(decoder_model.device)

with torch.inference_mode():
    output = decoder_model.generate(**inputs, max_new_tokens=120, temperature=0.7)

print("\n💬 Output:")
print(decoder_tokenizer.decode(output[0], skip_special_tokens=True))


ValueError: 
                    Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the
                    quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules
                    in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to
                    `from_pretrained`. Check
                    https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu
                    for more details.
                    

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

llama_name = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

decoder_tokenizer = AutoTokenizer.from_pretrained(llama_name, use_fast=True)
decoder_model = AutoModelForCausalLM.from_pretrained(
    llama_name,
    quantization_config=bnb_config,
    device_map="auto",                 # place layers automatically
    low_cpu_mem_usage=True,            # minimize GPU RAM spikes
    torch_dtype=torch.bfloat16,        # explicit compute type
)

decoder_tokenizer.pad_token = decoder_tokenizer.eos_token
print("✅ LLaMA 2 7B loaded successfully in 4-bit mode with CPU offload")

# 🔍 Quick sanity test
prompt = "Explain the role of transformers in modern NLP."
inputs = decoder_tokenizer(prompt, return_tensors="pt").to(decoder_model.device)

with torch.inference_mode():
    outputs = decoder_model.generate(**inputs, max_new_tokens=100, temperature=0.7)

print("\n💬 Output:")
print(decoder_tokenizer.decode(outputs[0], skip_special_tokens=True))


ValueError: 
                    Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the
                    quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules
                    in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to
                    `from_pretrained`. Check
                    https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu
                    for more details.
                    

In [3]:
# test_optimized_vedhllm.py
import sys
sys.path.append(".")

from modules.hybrid_model_optimized import VedhLLMOptimized
from modules.fusion_retriever import hybrid_search
from modules.symbolic_retriever import init_symbolic_db, insert_documents
from modules.semantic_retriever import build_semantic_index
import torch

print("🚀 Initializing optimized VedhLLM...")

# Initialize RAG
docs = [
    ("VedhLLM Overview", "VedhLLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 2025."),
    ("LLaMA Evolution", "LLaMA 3 outperformed GPT-4 in efficiency benchmarks."),
]

init_symbolic_db()
insert_documents(docs)
build_semantic_index()

# Initialize optimized model
model = VedhLLMOptimized(
    encoder_name="bert-base-uncased",
    decoder_name="meta-llama/Llama-2-7b-chat-hf",
    use_4bit=True,
    offload_encoder=True  # Key optimization
)

# Test query
query = "Who built VedhLLM?"
print(f"\n❓ Query: {query}")

# Retrieve
results = hybrid_search(query, top_k=2, verbose=False)
retrieved_docs = [(title, data["content"]) for title, data in results]

print(f"📄 Retrieved {len(retrieved_docs)} documents")

# Generate (decoder loads here)
print("💭 Generating answer...")
answer = model.generate(
    query=query,
    retrieved_docs=retrieved_docs,
    max_new_tokens=128,
    temperature=0.7,
    unload_after=True  # Free memory after generation
)

print(f"\n✅ Answer: {answer}")
print(f"\n💾 Final GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")



ModuleNotFoundError: No module named 'modules'